# Web application integration and technical evaluation
Application source: `deployment/frontend` (React, TypeScript, Vite/Node.js) and `deployment/backend` (FastAPI/Python), edited in VS Code.
Colab runs the ML workflow and verifies the saved model against the application API. No model conversion is used.
Run notebook 01 first. Upload the deployment source archive from the repository when prompted.

In [ ]:
import importlib.metadata as metadata
import subprocess, sys, os, signal
versions = {'numpy':'2.2.6','pandas':'2.2.3','scipy':'1.15.3','scikit-learn':'1.7.2','xgboost':'3.0.5','shap':'0.48.0','matplotlib':'3.10.3','joblib':'1.5.1','fastapi':'0.115.12','uvicorn':'0.34.2','pydantic':'2.11.7','pytest':'8.4.2','httpx':'0.28.1'}
changed = False
for package, version in versions.items():
    try:
        changed |= metadata.version(package) != version
    except metadata.PackageNotFoundError:
        changed = True
if changed:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*[f'{p}=={v}' for p,v in versions.items()]])
    print('Environment installed. Runtime is restarting; run all cells again after reconnection.', flush=True)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Pinned environment ready.')


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Final Research - C D Jayasinghe')

## 1. Import the application source and evaluated model

In [ ]:
import json, shutil, sys, subprocess, zipfile, io
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError('Upload deployment_source.zip only.')
APP_ROOT=Path('/content/h3_application')
APP_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for member in archive.infolist():
        target=(APP_ROOT/member.filename).resolve()
        if not target.is_relative_to(APP_ROOT.resolve()):
            raise ValueError('Unsafe archive member.')
    archive.extractall(APP_ROOT)
deployment=APP_ROOT/'deployment'
(deployment/'models').mkdir(parents=True,exist_ok=True)
for name in ['h3_pipeline.joblib','schema.json']:
    shutil.copy2(ROOT/'models'/name,deployment/'models'/name)
print('Application source and evaluated model ready.')

## 2. Functional and integration evaluation

In [ ]:
result=subprocess.run([sys.executable,'-m','pytest','-q','backend/test_api.py',
    '--junitxml='+str(ROOT/'outputs/app_tests.xml')],cwd=deployment,capture_output=True,text=True)
print(result.stdout)
(ROOT/'outputs/app_tests.txt').write_text(result.stdout,encoding='utf-8')
if result.returncode:
    print(result.stderr)
    raise RuntimeError('Application tests failed. Resolve failures before deployment.')

## 3. API response time

In [ ]:
import time
import pandas as pd
sys.path.insert(0,str(deployment))
from fastapi.testclient import TestClient
from backend.main import app
with TestClient(app) as client:
    assert client.get('/api/health').status_code==200
    payload={'eligible':True,'responses':dict.fromkeys([f'D{i:02d}' for i in range(1,21)],2)}
    client.post('/api/predict',json=payload).raise_for_status()
    durations=[]
    for _ in range(30):
        start=time.perf_counter()
        response=client.post('/api/predict',json=payload)
        response.raise_for_status()
        durations.append((time.perf_counter()-start)*1000)
latency=pd.Series(durations,name='api_ms')
latency.to_csv(ROOT/'outputs/api_latency_runs.csv',index=False)
summary=latency.quantile([0.5,0.95]).rename(index={0.5:'p50',0.95:'p95'})
summary.to_csv(ROOT/'outputs/api_latency_summary.csv');display(summary)
print('Includes inference, SHAP and API serialization in Colab; excludes browser, network and hosting cold start.')

## 4. Export the evaluated deployment artifacts
Download the bundle and extract it into the GitHub checkout. It contains the selected model and aggregate evaluation results, never raw responses. React type checking, production build and browser testing run from VS Code/the local checkout.

In [ ]:
bundle=ROOT/'outputs/deployment_bundle.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (deployment/'models').glob('*'):
        if path.is_file(): archive.write(path,'deployment/models/'+path.name)
    aggregate=['data_audit.json','split_summary.csv','cv_comparison.csv','test_comparison.csv',
        'model_selection.json','selected_model_confidence_intervals.csv','shap_global.csv',
        'app_tests.xml','app_tests.txt','api_latency_summary.csv','environment.json']
    for name in aggregate:
        path=ROOT/'outputs'/name
        if path.is_file(): archive.write(path,'evaluation/'+name)
    for path in (ROOT/'report_figures').glob('*.png'):
        archive.write(path,'evaluation/figures/'+path.name)
files.download(str(bundle))

## 5. Hosting verification
Use the repository's Dockerfile and `render.yaml` on Render's free web-service plan. Deploy only after the React build, API tests, and real-model integration checks pass.
Set the deployed URL below, then complete hosted browser acceptance, user usability and expert review in notebook 03. Free instances can sleep after inactivity; measure cold-start and warm response time separately.

In [ ]:
APP_URL=''
if APP_URL:
    import requests
    from urllib.parse import urlparse
    parsed=urlparse(APP_URL)
    if parsed.scheme!='https' or not parsed.hostname or not parsed.hostname.endswith('.onrender.com'):
        raise ValueError('Use the deployed Render HTTPS URL.')
    response=requests.get(APP_URL.rstrip('/')+'/api/health',timeout=120)
    response.raise_for_status()
    assert response.json()['status']=='ok'
    (ROOT/'outputs/hosting_check.json').write_text(json.dumps({'url':APP_URL,'status':response.status_code,'browser_acceptance':'pending'}))
    print('API health passed; complete hosted browser acceptance.')
else:
    print('Hosting check pending: enter the deployed URL.')